# 05. Skill Optimization — dialog-summary SKILL.md
Optimize `dialog-summary/SKILL.md` against samsum gold summaries via Arize experiments.

In [1]:
!pip install -qqq arize arize-phoenix certifi urllib3 langchain-openai python-dotenv

In [2]:
import os, certifi
CA = certifi.where()
os.environ["SSL_CERT_FILE"] = CA
os.environ["REQUESTS_CA_BUNDLE"] = CA
os.environ["CURL_CA_BUNDLE"] = CA

from arize import ArizeClient
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv("ARIZE_API_KEY")
SPACE_ID = os.getenv("ARIZE_SPACE_ID")
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set (not in .env) — export it before running"

client = ArizeClient(api_key=API_KEY, request_verify=False)

# ---- demo config (single source of truth) ----
N_EXAMPLES   = 50
N_ITERATIONS = 3
GEN_MODEL    = "gpt-4.1-2025-04-14"
JUDGE_MODEL  = "gpt-4o-mini"
SKILL_PATH   = "dialog-summary/SKILL.md"
VERSIONS_DIR = "dialog-summary/versions"
DATASET_NAME = "samsum_small"

In [3]:
examples = client.datasets.list_examples(dataset=DATASET_NAME, space=SPACE_ID)
examples

  arize.pre_releases | WARNING | [BETA] datasets.list_examples is a beta API in Arize SDK v8.37.1 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


/Users/sean/projects/healingpaper/.venv/lib/python3.13/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.arize.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


DatasetExampleListResponse(examples=[DatasetExample(id='00f72bad-ab78-4d51-9df7-e4be79905611', created_at=datetime.datetime(2026, 6, 26, 3, 25, 16, tzinfo=TzInfo(0)), updated_at=datetime.datetime(2026, 6, 26, 3, 25, 16, tzinfo=TzInfo(0)), annotations=None, additional_properties={'dialogue': "Mary: Sorry, I didn't make it to your bday party :(\nNick: It's OK...\nMary: But I just got SOOO distracted! I forgot it was yesterday!\nNick: do tell!\nMary: I met this guy...\nNick: REALLY? I want details :D\nMary: Yeah, his name is Kirk and he's an architect...\nNick: OK, just your type then <file_gif>\nMary: And we ended up spending the whole week together. xD\nNick: A WEEK?\nMary: Yeah... It's madness, I'll tell you more this evening. Are we still on?\nNick: You bet we are!", 'summary': "Mary didn't come to Nick's birthday party. She met an architect named Kirk. Mary and Nick will meet in the evening."}), DatasetExample(id='02a40690-2e21-403c-b883-26e1b263e4d9', created_at=datetime.datetime(20

In [4]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

gen_llm = ChatOpenAI(model=GEN_MODEL, temperature=0.0)
gen_chain = gen_llm | StrOutputParser()

def read_skill() -> str:
    with open(SKILL_PATH, "r") as f:
        return f.read()

def run_skill(dataset_row) -> str:
    """Use the full SKILL.md text as the system prompt over one dialogue."""
    skill = read_skill()
    dialogue = dataset_row["dialogue"]
    messages = [
        ("system", skill),
        ("human", f"Summarize this dialogue:\n\n{dialogue}"),
    ]
    return gen_chain.invoke(messages)

In [5]:
# smoke test on one example (does NOT hit Arize)
_sample = examples.to_df().iloc[0] if hasattr(examples, "to_df") else pd.DataFrame(examples).iloc[0]
print("DIALOGUE:\n", _sample["dialogue"][:400])
print("\nGOLD:\n", _sample["summary"])
print("\nSKILL OUTPUT:\n", run_skill(_sample))

DIALOGUE:
 Mary: Sorry, I didn't make it to your bday party :(
Nick: It's OK...
Mary: But I just got SOOO distracted! I forgot it was yesterday!
Nick: do tell!
Mary: I met this guy...
Nick: REALLY? I want details :D
Mary: Yeah, his name is Kirk and he's an architect...
Nick: OK, just your type then <file_gif>
Mary: And we ended up spending the whole week together. xD
Nick: A WEEK?
Mary: Yeah... It's madness,

GOLD:
 Mary didn't come to Nick's birthday party. She met an architect named Kirk. Mary and Nick will meet in the evening.



SKILL OUTPUT:
 Mary apologizes to Nick for missing his birthday party because she was distracted after meeting someone new, Kirk, with whom she spent the week. She promises to share more details when they meet later that evening, and Nick confirms their plans.
